In [1]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle

# Download stopwords
nltk.download('stopwords')

# Load dataset
dataset = pd.read_csv("1000_dataset_english (2).csv", encoding="utf-8")

# Display first few rows
print(dataset.head())


        PostID                                   Post Description        Date  \
0  B7mbLCVhYIf  QUESTIONS AND ANSWERS ON CORONAVIRUS PT. 2\n\n...  01/21/2020   
1  B7m7M3SgvI1  Using humor to bring attention to a serious ma...  01/22/2020   
2  B7oK_DMhtxr  Stay safe\nCover your face\n#typ262 #40mm #40m...  01/22/2020   
3  B7njeFbBX5y  Did you know an outbreak of a new coronavirus ...  01/22/2020   
4  B7nV26QgOLP  It’s so sad and scary that the coronavirus has...  01/22/2020   

  Language Code Full Language Sentiment  
0            en       English  positive  
1            en       English  negative  
2            en       English  positive  
3            en       English  negative  
4            en       English  negative  


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Uday\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Check missing values
print(dataset.isnull().sum())


PostID              0
Post Description    0
Date                0
Language Code       0
Full Language       0
Sentiment           0
dtype: int64


In [3]:

# Manually specify the column names
column_names = ["PostID", "Post Description", "Date", "Language Code", "Full Language", "Sentiment"]

# Read CSV properly
dataset = pd.read_csv("1000_dataset_english (2).csv", names=column_names, skiprows=1)

# Display the first few rows
print(dataset.head())


        PostID                                   Post Description        Date  \
0  B7mbLCVhYIf  QUESTIONS AND ANSWERS ON CORONAVIRUS PT. 2\n\n...  01/21/2020   
1  B7m7M3SgvI1  Using humor to bring attention to a serious ma...  01/22/2020   
2  B7oK_DMhtxr  Stay safe\nCover your face\n#typ262 #40mm #40m...  01/22/2020   
3  B7njeFbBX5y  Did you know an outbreak of a new coronavirus ...  01/22/2020   
4  B7nV26QgOLP  It’s so sad and scary that the coronavirus has...  01/22/2020   

  Language Code Full Language Sentiment  
0            en       English  positive  
1            en       English  negative  
2            en       English  positive  
3            en       English  negative  
4            en       English  negative  


In [4]:
print(dataset.columns)
print(dataset.shape)


Index(['PostID', 'Post Description', 'Date', 'Language Code', 'Full Language',
       'Sentiment'],
      dtype='object')
(999, 6)


In [5]:
import nltk
from nltk.corpus import stopwords
import re

# Download stopwords if not already available
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if isinstance(text, str):  # Ensure text is string
        text = text.lower()  # Lowercase
        text = re.sub(r'http\S+', '', text)  # Remove URLs
        text = re.sub(r'[^a-z\s]', '', text)  # Remove special characters
        text = ' '.join([word for word in text.split() if word not in stop_words])  # Remove stopwords
        return text
    return ""

# Apply processing to 'Post Description'
dataset['Processed_Text'] = dataset['Post Description'].apply(preprocess_text)

# Verify
print(dataset[['Post Description', 'Processed_Text']].head())  # Check processed text


                                    Post Description  \
0  QUESTIONS AND ANSWERS ON CORONAVIRUS PT. 2\n\n...   
1  Using humor to bring attention to a serious ma...   
2  Stay safe\nCover your face\n#typ262 #40mm #40m...   
3  Did you know an outbreak of a new coronavirus ...   
4  It’s so sad and scary that the coronavirus has...   

                                      Processed_Text  
0  questions answers coronavirus pt vaccine coron...  
1  using humor bring attention serious matter com...  
2  stay safe cover face typ mm mmperspective face...  
3  know outbreak new coronavirus china killed nin...  
4  sad scary coronavirus killed sickened many peo...  


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Uday\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, chi2

# Vectorize text using TF-IDF
# Remove neutral sentiment from the dataset
dataset = dataset[dataset['Sentiment'] != 'neutral']

# Now continue with feature extraction
vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1,3), stop_words='english')
X = vectorizer.fit_transform(dataset['Processed_Text']).toarray()
y = dataset['Sentiment']

# Check label distribution
print(y.value_counts())  # Should now show only "positive" and "negative"


# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train Random Forest model with optimized hyperparameters
model = RandomForestClassifier(n_estimators=300, max_depth=30, min_samples_split=5, min_samples_leaf=2, 
                               random_state=42, class_weight="balanced_subsample", bootstrap=True)  #class_weight: helps with imbalanced data
 # Enable bootstrapping

model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Check accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")


Sentiment
positive    354
negative    323
Name: count, dtype: int64
Model Accuracy: 0.72


In [7]:
print(dataset['Sentiment'].value_counts(normalize=True))  # Show class balance


Sentiment
positive    0.522895
negative    0.477105
Name: proportion, dtype: float64


In [8]:
import pickle

# Save the trained model
pickle.dump(model, open('model.pkl', 'wb'))

# Save the TF-IDF vectorizer
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))

print("Model and Vectorizer Saved!")


Model and Vectorizer Saved!


In [9]:
# Select random samples from the dataset for testing
sample_data = dataset.sample(n=5, random_state=42)  # Pick 5 random samples
sample_texts = sample_data['Processed_Text'].tolist()

# Transform input text using the same vectorizer
sample_features = vectorizer.transform(sample_texts).toarray()

# Predict sentiment
predictions = model.predict(sample_features)

# Display results
for text, sentiment in zip(sample_texts, predictions):
    print(f"Text: {text}\nPredicted Sentiment: {sentiment}\n")


Text: credit coronavirusonline coronavirus coronavirusoutbreak virus wuhan wuhanvirus coronaviruschina wuhanchina corona coronovirus chinavirus wuhancoronavirus outbreak plagueinc corona coronaviruses coronavirusfrance coronavirus coronavirusbrasil coronavirusnews coronavirusecuador coronaviruswuhan coronavirusindia coronavirusupdate coronavirusawareness coronavirusdewuhan coronavirusoutbreak coronavirusaustralia coronavirus coronavirusu
Predicted Sentiment: positive

Text: got opportunity host programme doordarshan interviewed dr ak dwivedi spreading awareness corona virus coronavirusawareness homeopathy tvhost doordarshan
Predicted Sentiment: positive

Text: heart goes everyone affected coronavirus stay safe everyone wash coronaviruschina plaugemarines prilaga coronavirus coronavirus coronavirusmemes coronavrus coronavirs plaugeoc plaugeinc plaugeaffiliates plaugedoctormask coronavirusawareness coronaviruses coronavirusnews plaugedoctormemes coronaviruschino plaugemask plaugedoctor c

In [10]:
import numpy as np

# Get feature importance
feature_importances = model.feature_importances_
top_n = 20  # Number of top features to display

# Get top features
top_features_idx = np.argsort(feature_importances)[::-1][:top_n]
top_features = [vectorizer.get_feature_names_out()[i] for i in top_features_idx]

print("Top Important Features in Sentiment Classification:")
print(top_features)


Top Important Features in Sentiment Classification:
['safe', 'coronavirus', 'coronavirusawareness', 'death', 'wuhanvirus', 'stay', 'coronaviruschina', 'outbreak', 'healthy', 'coronavirusoutbreak', 'chinavirus', 'coronavirusupdates', 'avoid', 'wuhan', 'death toll', 'virus', 'protect', 'thank', 'sick', 'infected']
